# 📊 Pocket OTC AI Analyzer — Google Colab

يشغّل واجهة الويب وTelegram Bot معًا.

**READ-ONLY:** لا يسجل الدخول إلى Pocket Option ولا ينفذ أوامر تداول.

In [ ]:
!rm -rf /content/Jjjjjjj
!git clone -q https://github.com/mohmb142/Jjjjjjj.git /content/Jjjjjjj
%cd /content/Jjjjjjj
!python -m pip install -q -r colab_requirements.txt
print('✅ تم تنزيل المشروع وتثبيت المتطلبات')

In [ ]:
import os
from getpass import getpass

telegram_token = getpass('🔐 Telegram Bot Token: ').strip()
openrouter_key = getpass('🔐 OpenRouter API Key: ').strip()

if not telegram_token:
    raise ValueError('Telegram Bot Token مطلوب')
if not openrouter_key:
    raise ValueError('OpenRouter API Key مطلوب')

os.environ['TELEGRAM_BOT_TOKEN'] = telegram_token
os.environ['OPENROUTER_API_KEY'] = openrouter_key
os.environ['OPENROUTER_MODEL'] = 'google/gemini-2.5-flash'
print('✅ تم إعداد المفاتيح داخل جلسة Colab فقط')

In [ ]:
# تشغيل Web UI + Telegram Bot معًا
import socket
import threading
import time
import uvicorn
from google.colab.output import eval_js
from IPython.display import HTML, display

PORT = 8000

def port_ready(host='127.0.0.1', port=PORT, timeout=30):
    end = time.time() + timeout
    while time.time() < end:
        try:
            with socket.create_connection((host, port), timeout=0.5):
                return True
        except OSError:
            time.sleep(0.25)
    return False

def web_worker():
    try:
        uvicorn.run('main:app', host='0.0.0.0', port=PORT, log_level='warning')
    except Exception as exc:
        print('❌ Web UI error:', exc)

if 'web_thread' not in globals() or not web_thread.is_alive():
    web_thread = threading.Thread(target=web_worker, daemon=True, name='fastapi-web')
    web_thread.start()

if not port_ready():
    raise RuntimeError('❌ لم تبدأ FastAPI على المنفذ 8000')

public_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')

html = f'''
<div style="padding:24px;border:1px solid #ddd;border-radius:12px;font-family:Arial,sans-serif;margin:10px 0">
<h2>📊 Pocket OTC AI Analyzer</h2>
<p>✅ واجهة تحليل صور الشارت تعمل الآن.</p>
<a href="{public_url}" target="_blank" style="display:inline-block;padding:14px 24px;background:#1976d2;color:#fff;text-decoration:none;border-radius:8px;font-weight:bold">🚀 فتح الواجهة</a>
<p style="margin-top:15px;word-break:break-all"><b>الرابط العام:</b><br>{public_url}</p>
</div>
'''
display(HTML(html))
print('🌐 الرابط العام:', public_url)

from telegram_bot import run

def telegram_worker():
    try:
        print('🤖 بدء Telegram Bot...')
        run()
    except Exception as exc:
        print('❌ Telegram Bot error:', exc)

if 'telegram_thread' not in globals() or not telegram_thread.is_alive():
    telegram_thread = threading.Thread(target=telegram_worker, daemon=True, name='telegram-bot')
    telegram_thread.start()

print('🤖 Telegram Bot: يعمل في الخلفية')
print('🌐 Web UI: تعمل في الخلفية')
print('✅ Web UI + Telegram يعملان معًا')

In [ ]:
# اختبار الواجهة
import requests

response = requests.get('http://127.0.0.1:8000/health', timeout=10)
print('HTTP status:', response.status_code)
if response.status_code != 200:
    raise RuntimeError('❌ فشل اختبار FastAPI')
print('✅ FastAPI health check نجح')
print('🌐 الرابط العام:', public_url)
print('⚠️ لا تغلق جلسة Colab حتى يبقى الرابط والبوت يعملان.')